# Support Vector Machine — Training and Analysis





Trains a linear SVM on the **leakage-corrected** CICEVSE2024 network-traffic


dataset.





| | |


|---|---|


| **Dataset** | CICEVSE2024 network traffic — 15 classes (14 attacks + benign) |


| **Features** | 60, after dropping six capture timestamps and `src_port` |


| **Task** | Multiclass only |


| **Headline metric** | Macro-F1 |





> **Binary classification was removed from this notebook.** The dataset contains


> **82 benign flows in total**, so an attack-vs-benign task is answered by


> predicting "attack" every time. `tests/test_epic2_compliance.py` asserts no


> `*_binary.pkl` artifact exists, and the previous version of this notebook wrote


> one — running it would have failed the suite.





> **The SVM is the control in the leakage story.** Removing the timestamps barely


> moved it (0.2829 → 0.3084 macro-F1) while Random Forest fell 0.9547 → 0.5894.


> A hyperplane cannot exploit a timestamp axis the way a tree can — which is what


> the old 0.9994-vs-0.3665 gap was really measuring, not non-linearity.





`LinearSVC` on the full matrix exhausted memory during the original work, which is


why `src/models/svm/train_svm.py` uses `SGDClassifier(loss="hinge")` with


`partial_fit` over chunks. This notebook subsamples instead, for interactivity.

In [ ]:
import os



import joblib



import numpy as np



import pandas as pd



import matplotlib.pyplot as plt



import seaborn as sns







from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report







# Plotting config



%matplotlib inline



plt.rcParams['figure.dpi'] = 100



sns.set_theme(style='whitegrid', palette='deep')

## 1. Load Data



Load the preprocessed train, validation, and test datasets. Note: ensure `preprocess.py` has been run previously.

In [ ]:
# Adjust path assuming the notebook runs from the project root or src/models/svm

import sys

if os.path.exists('../../../data/processed'):

    DATA_DIR = '../../../data/processed'

elif os.path.exists('data/processed'):

    DATA_DIR = 'data/processed'

else:

    DATA_DIR = '../data/processed' # fallback



print(f"Using data directory: {DATA_DIR}")



X_train = pd.read_csv(os.path.join(DATA_DIR, "X_train.csv"))

y_train = pd.read_csv(os.path.join(DATA_DIR, "y_train.csv"))

X_val = pd.read_csv(os.path.join(DATA_DIR, "X_val.csv"))

y_val = pd.read_csv(os.path.join(DATA_DIR, "y_val.csv"))

X_test = pd.read_csv(os.path.join(DATA_DIR, "X_test.csv"))

y_test = pd.read_csv(os.path.join(DATA_DIR, "y_test.csv"))





y_train_multi = y_train["Label_Multiclass"].values.ravel()



y_val_multi = y_val["Label_Multiclass"].values.ravel()



y_test_multi = y_test["Label_Multiclass"].values.ravel()



print("Data loaded successfully!")

print(f"X_train shape: {X_train.shape}")

In [ ]:
# ── Leakage guard (issues #46, #58) ──────────────────────────────────────────
# The published CICEVSE2024 pipeline keeps six absolute capture timestamps that
# identify the *recording* rather than the traffic: a decision tree on
# `bidirectional_first_seen_ms` alone scores 1.0000 on the 15-class target.
# `src_port` is a weaker version of the same thing — ephemeral ports are
# allocated near-sequentially, so they band per capture (0.4028 alone).
#
# Both are dropped by `preprocess.py`. This cell fails loudly if you are sitting
# on data built by an older version of the pipeline, because every number below
# would be meaningless.
LEAKING = [
    "bidirectional_first_seen_ms", "bidirectional_last_seen_ms",
    "src2dst_first_seen_ms", "src2dst_last_seen_ms",
    "dst2src_first_seen_ms", "dst2src_last_seen_ms",
    "src_port",
]

# Notebooks load either the full matrix or a sample frame for visualisation.
_frame = next((globals()[n] for n in ("X_train", "X_train_sample") if n in globals()), None)
assert _frame is not None, "Run the data-loading cell above first."

present = [c for c in LEAKING if c in _frame.columns]
assert not present, (
    f"Leaking columns found: {present}. Regenerate with: make data-process"
)
print(f"Leakage guard passed — {_frame.shape[1]} features, none of them capture fingerprints.")

# Relative timing is behavioural and is deliberately retained.
kept = [c for c in _frame.columns if c.endswith("_duration_ms") or c.endswith("_piat_ms")]
print(f"Relative timing features retained: {len(kept)}")


## 1.1 Dataset Overview



Quick inspection of the training data: shape, data types, summary statistics, and missing value check.

In [ ]:
print(f"X_train shape : {X_train.shape}")



print(f"X_val shape   : {X_val.shape}")



print(f"X_test shape  : {X_test.shape}")



print(f"\nNumber of features: {X_train.shape[1]}")



print(f"\nData types:\n{X_train.dtypes.value_counts()}")



print(f"\nMissing values per column (if any):")



missing = X_train.isnull().sum()



missing_cols = missing[missing > 0]



if len(missing_cols) == 0:



    print("  None — all features are clean.")



else:



    print(missing_cols)







print("\nSummary Statistics (first 10 features):")



X_train.iloc[:, :10].describe().round(3)

## 1.2 Train / Validation / Test Split Sizes



Visualise the data split proportions to verify the 70/15/15 split from preprocessing.

In [ ]:
split_sizes = pd.DataFrame({



    'Split': ['Train', 'Validation', 'Test'],



    'Samples': [len(X_train), len(X_val), len(X_test)]



})



split_sizes['Percentage'] = (split_sizes['Samples'] / split_sizes['Samples'].sum() * 100).round(1)







fig, ax = plt.subplots(figsize=(8, 3))



bars = ax.barh(split_sizes['Split'], split_sizes['Samples'], color=['#2196F3', '#FF9800', '#4CAF50'])



for bar, pct in zip(bars, split_sizes['Percentage']):



    ax.text(bar.get_width() + 5000, bar.get_y() + bar.get_height()/2,



            f'{bar.get_width():,.0f} ({pct}%)', va='center', fontsize=11)



ax.set_xlabel('Number of Samples')



ax.set_title('Train / Validation / Test Split Sizes')



plt.tight_layout()



plt.show()







print(split_sizes.to_string(index=False))

## 1.3 Class distribution



Multiclass only. The binary target is not modelled here: the dataset holds 82

benign flows against 1.2M attack flows, so "attack vs benign" is answered by

predicting attack every time.

In [ ]:
multi_counts = y_train["Label_Multiclass"].value_counts()



# Colour by family, because the volumetric/reconnaissance split is the finding.

VOLUMETRIC = {"SYN_Flood", "TCP_Flood", "UDP_Flood", "SynonymousIP_Flood", "PSHACK_Flood"}

RECON = {"TCP_Port_Scan", "SYN_Stealth_Scan", "Service_Version_Detection",

         "OS_Fingerprinting", "Aggressive_Scan", "Vulnerability_Scan"}

def family(c): return "volumetric" if c in VOLUMETRIC else "recon" if c in RECON else "other"

PALETTE = {"volumetric": "#2E7D32", "recon": "#EF6C00", "other": "#607D8B"}



colors = [PALETTE[family(c)] for c in multi_counts.index]



fig, ax = plt.subplots(figsize=(12, 6))

ax.barh(multi_counts.index[::-1], multi_counts.values[::-1], color=colors[::-1])

ax.set_title("Class distribution (train) — coloured by attack family", fontsize=14)

ax.set_xlabel("Flows")

for i, val in enumerate(multi_counts.values[::-1]):

    ax.text(val * 1.01, i, f"{val:,}", va="center", fontsize=9)

handles = [plt.Rectangle((0, 0), 1, 1, color=v) for v in PALETTE.values()]

ax.legend(handles, PALETTE.keys(), loc="lower right", frameon=True)

plt.tight_layout(); plt.show()



print(multi_counts.to_string())

print(f"\nBenign flows in train: {int((y_train['Label_Multiclass'] == 'Benign').sum())}")

## 1.4 Feature Correlation Heatmap



Visualise the pairwise Pearson correlations of the top 30 features (by variance) to identify multicollinearity.

In [ ]:
# Select top 30 features by variance for readability



top_features = X_train.var().nlargest(30).index.tolist()



corr_matrix = X_train[top_features].corr()







plt.figure(figsize=(14, 12))



mask = np.triu(np.ones_like(corr_matrix, dtype=bool))



sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', center=0,



            square=True, linewidths=0.5, fmt='.1f',



            cbar_kws={'shrink': 0.8, 'label': 'Pearson Correlation'})



plt.title('Feature Correlation Heatmap (Top 30 by Variance)', fontsize=14)



plt.xticks(rotation=45, ha='right', fontsize=8)



plt.yticks(fontsize=8)



plt.tight_layout()



plt.show()

## 1.5 Feature Distribution Box Plots



Box plots of the top 10 highest-variance features to visualise their range and spread after StandardScaler normalisation.

In [ ]:
top10 = X_train.var().nlargest(10).index.tolist()







fig, axes = plt.subplots(2, 5, figsize=(20, 8))



axes = axes.flatten()







for i, col in enumerate(top10):



    # Sample for speed (1% of data is enough for distribution shape)



    sample = X_train[col].sample(n=min(10000, len(X_train)), random_state=42)



    axes[i].boxplot(sample.values, vert=True, patch_artist=True,



                    boxprops=dict(facecolor='#42A5F5', alpha=0.7))



    axes[i].set_title(col, fontsize=9, fontweight='bold')



    axes[i].tick_params(axis='x', labelbottom=False)







plt.suptitle('Top 10 Features by Variance — Box Plots (Scaled Data)', fontsize=14, y=1.02)



plt.tight_layout()



plt.show()

## 2. Train the multiclass SVM



`LinearSVC` on the full 838k x 60 matrix exhausted memory during the original

work, which is why `src/models/svm/train_svm.py` uses

`SGDClassifier(loss="hinge")` with `partial_fit` over chunks — mathematically the

same linear SVM, trained out-of-core.



This notebook subsamples so it stays interactive. For the production fit, run the

script.

In [ ]:
from sklearn.linear_model import SGDClassifier

from sklearn.utils.class_weight import compute_class_weight



# Subsample for interactivity; the script trains on all 838k rows.

SUBSAMPLE = 200_000

idx = np.random.RandomState(42).choice(len(X_train), min(SUBSAMPLE, len(X_train)), replace=False)

X_fit, y_fit = X_train.iloc[idx], pd.Series(y_train_multi).iloc[idx]



classes = np.array(sorted(y_fit.unique()))

class_weight = dict(zip(classes, compute_class_weight("balanced", classes=classes, y=y_fit)))



print(f"Fitting on {len(X_fit):,} rows across {len(classes)} classes...")

best_multi_model = SGDClassifier(loss="hinge", random_state=42, class_weight=class_weight)

best_multi_model.fit(X_fit, y_fit)

print("Done.")

## 3. Evaluation Setup

In [ ]:
def evaluate_model(model, X, y, title_prefix=""):
    """
    Report macro-F1 first.

    Accuracy on this dataset is dominated by the volumetric floods, which every
    model solves, so it compresses genuinely different models into the same
    number. Macro-F1 weights each class equally and is what the write-up quotes.
    """
    preds = model.predict(X)

    acc = accuracy_score(y, preds)
    f1_macro = f1_score(y, preds, average="macro", zero_division=0)
    f1_weighted = f1_score(y, preds, average="weighted", zero_division=0)

    print(f"--- {title_prefix} ---")
    print(f"Macro F1-Score : {f1_macro:.4f}   <- headline")
    print(f"Weighted F1    : {f1_weighted:.4f}")
    print(f"Accuracy       : {acc:.4f}   (inflated by the flood classes)")
    print("\nClassification report:")
    print(classification_report(y, preds, zero_division=0))

    labels = sorted(np.unique(np.concatenate([np.asarray(y), np.asarray(preds)])))
    cm = confusion_matrix(y, preds, labels=labels)
    size = max(8, len(labels) * 0.75)
    plt.figure(figsize=(size + 2, size))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels, cbar_kws={"shrink": 0.7})
    plt.title(f"{title_prefix} — confusion matrix")
    plt.ylabel("Actual"); plt.xlabel("Predicted")
    plt.xticks(rotation=45, ha="right", fontsize=8); plt.yticks(fontsize=8)
    plt.tight_layout(); plt.show()
    return preds


## 4. Evaluate



Macro-F1 leads. Accuracy on this dataset is dominated by the volumetric floods

and would rank every model here almost identically.

In [ ]:
_ = evaluate_model(best_multi_model, X_val, y_val_multi, title_prefix="Validation")

## 5. Test-set evaluation

In [ ]:
_ = evaluate_model(best_multi_model, X_test, y_test_multi, title_prefix="Test")

## 6. Save the model



Only the multiclass model is written. `tests/test_epic2_compliance.py` asserts

that no `*_binary.pkl` artifact exists.

In [ ]:
if os.path.exists("../../../saved_models"):

    SAVE_DIR = "../../../saved_models"

elif os.path.exists("saved_models"):

    SAVE_DIR = "saved_models"

else:

    SAVE_DIR = "../saved_models"



os.makedirs(SAVE_DIR, exist_ok=True)

joblib.dump(best_multi_model, os.path.join(SAVE_DIR, "svm_model_multiclass.pkl"))

print(f"Saved svm_model_multiclass.pkl to {SAVE_DIR}")

---









## Where the canonical numbers live









This notebook is for exploration. Every figure quoted in the write-up comes from




`src/evaluation/run_experiments.py`, which re-runs the split *and* model




initialisation across seeds 42 / 1337 / 2024 and reports mean ± standard




deviation. A single-seed result from this notebook will differ, and should not




be quoted on its own.









```bash




python3 src/evaluation/run_experiments.py --raw-dir data/raw --feature-set extended




```









| Reference | Location |




|---|---|




| Multi-seed results | `evaluation_results/multiseed/MULTISEED_RESULTS.md` |




| Full leakage audit | `docs/leakage_audit_results.md` |




| Feature-set diff vs. the reference implementation | `docs/dataset_feature_engineering.md` §3.4 |









**Headline for this dataset is macro-F1, not accuracy.** With `ICMP_Fragmentation`




at 28 flows and `Benign` at 82, accuracy tracks the volumetric floods and hides




the reconnaissance classes almost entirely — the corrected Random Forest scores




0.8679 accuracy against 0.5582 macro-F1 on the same predictions.